# 01 · ETL — Clean raw Amazon reviews & metadata

Reads the raw Parquet files from `gs://msca-bdp-data-open/final_project_reviews` (~52 GiB), keeps only the columns needed downstream, and writes cleaned Parquet batches to Google Drive.

## Environment

In [ ]:
!pip install pyspark gcsfs -q
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

!wget -q https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar

from google.colab import drive, auth
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

auth.authenticate_user()

In [ ]:
from pyspark.sql import SparkSession

jar_path = "/content/gcs-connector-hadoop3-latest.jar"

spark = SparkSession.builder \
    .appName("Amazon_Reviews_ETL") \
    .config("spark.jars", jar_path) \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# E

In [ ]:
# I have investigated the Schema before, so it is possible for me to
# clean and save the parquet simultaneously.
'''
--- Reviews Schema ---
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)


--- Meta Schema ---
root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: integer (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- price: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- title: string (nullable = true)

In [ ]:
import gcsfs

BUCKET = "msca-bdp-data-open"
GCS_META_PATH = f"{BUCKET}/final_project_reviews/meta_parquet"
GCS_REVIEW_PATH = f"{BUCKET}/final_project_reviews/reviews_parquet"

BASE_DIR = "/content/drive/My Drive/BDCC_Final_Project"
META_OUT_DIR = f"{BASE_DIR}/cleaned_meta_spark"
REVIEW_OUT_DIR = f"{BASE_DIR}/cleaned_review_spark"

for d in [META_OUT_DIR, REVIEW_OUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

fs = gcsfs.GCSFileSystem()
raw_meta_files = fs.glob(f"{GCS_META_PATH}/*.parquet")
raw_review_files = fs.glob(f"{GCS_REVIEW_PATH}/*.parquet")

meta_files = [f"gs://{f}" for f in raw_meta_files]
review_files = [f"gs://{f}" for f in raw_review_files]

print(f"Meta files found: {len(meta_files)}")
print(f"Review files found: {len(review_files)}")

Meta files found: 50
Review files found: 50


## Meta

In [ ]:
from pyspark.sql.functions import col

META_BATCH_SIZE = 10

print("Starting Meta Data Processing...")

for i in range(0, len(meta_files), META_BATCH_SIZE):
    batch_files = meta_files[i : i + META_BATCH_SIZE]
    batch_id = (i // META_BATCH_SIZE) + 1

    save_path = f"{META_OUT_DIR}/batch_{batch_id:03d}"

    if os.path.exists(save_path):
        print(f"Skipping Batch {batch_id}, already exists.")
        continue

    print(f"Processing Meta Batch {batch_id}...")

    try:
        df_meta = spark.read.parquet(*batch_files)

        df_clean = df_meta.select(
            col("parent_asin"),
            col("title").alias("product_title"),
            col("main_category"),
            col("price")
        ).filter(
            col("main_category").isNotNull()
        ).dropDuplicates(["parent_asin"])

        df_clean.coalesce(1).write.mode("overwrite").parquet(save_path)

        print(f"Batch {batch_id} saved successfully.")

    except Exception as e:
        print(f"Error processing Meta Batch {batch_id}: {e}")

print("Meta Data Processing Complete.")

Starting Meta Data Processing...
Processing Meta Batch 1...
Batch 1 saved successfully.
Processing Meta Batch 2...
Batch 2 saved successfully.
Processing Meta Batch 3...
Batch 3 saved successfully.
Processing Meta Batch 4...
Batch 4 saved successfully.
Processing Meta Batch 5...
Batch 5 saved successfully.
Meta Data Processing Complete.


In [ ]:
from pyspark.sql.functions import from_unixtime, to_date, year, coalesce, lit, col

REVIEW_BATCH_SIZE = 5

FINAL_REVIEW_COLS = ["parent_asin", "rating", "title", "text", "timestamp", "user_id"]

print("Starting Review Data Processing...")

for i in range(0, len(review_files), REVIEW_BATCH_SIZE):
    batch_files = review_files[i : i + REVIEW_BATCH_SIZE]
    batch_id = (i // REVIEW_BATCH_SIZE) + 1

    save_path = f"{REVIEW_OUT_DIR}/batch_{batch_id:03d}"

    if os.path.exists(save_path):
        print(f"Skipping Batch {batch_id}, already exists.")
        continue

    print(f"Processing Batch {batch_id}...")

    try:
        df_rev = spark.read.parquet(*batch_files).select(*FINAL_REVIEW_COLS)
        df_clean = df_rev.withColumn(
            "review_date", to_date(from_unixtime(col("timestamp") / 1000))
        ).withColumn(
            "year", year(col("review_date"))
        ).withColumn(
            "text", coalesce(col("text"), lit(""))
        ).withColumn(
            "title", coalesce(col("title"), lit(""))
        ).filter(
            col("rating").isNotNull()
        ).select(
            "parent_asin",
            "user_id",
            "rating",
            "title",
            "text",
            "review_date",
            "year"
        )

        df_clean.coalesce(1).write.mode("overwrite").parquet(save_path)

        print(f"Batch {batch_id} saved.")

    except Exception as e:
        print(f"Error in batch {batch_id}: {e}")

print("Review Data Processing Complete.")

Starting Review Data Processing...
Processing Batch 1...
Batch 1 saved.
Processing Batch 2...
Batch 2 saved.
Processing Batch 3...
Batch 3 saved.
Processing Batch 4...
Batch 4 saved.
Processing Batch 5...
Batch 5 saved.
Processing Batch 6...
Batch 6 saved.
Processing Batch 7...
Batch 7 saved.
Processing Batch 8...
Batch 8 saved.
Processing Batch 9...
Batch 9 saved.
Processing Batch 10...
Batch 10 saved.
Review Data Processing Complete.
